# 风电场偏航优化 Nature 级科研绘图工厂

本 Notebook 使用 `scipilot-figure-skill` + `nature-figure` 两个顶级技能族，严格按 Nature 期刊规范（单栏 89mm=3.5in，双栏 183mm=7.2in，7-9pt 字号，色盲安全，矢量 PDF）批量出图。

**数据全景**：
- `cases.csv` 13 偏航工况 (-30~30° @8m/s) 串列 2 机：P1/P2/Ptot
- `cases_multi.csv` 4 风速 ×13 偏航=52 工况：增益矩阵
- `cases_array.csv` 3×3 九机阵列统一偏航 13 工况：9 机功率 + 总功率 + gain
- `array_independent_result.json` 独立贪心 [30°,20°,0°] vs 统一 30° vs 基线 0°：总功率 8095→9299→10041 kW (+14.87%→+24.04%)
- `fields/*.npz` 13 个 hub 高度 2D 流场切片 (128×64)
- `fields_array/*.npz` baseline vs independent 3×3 流场
- `pod_results/pod_data.npz` POD 10 模态，energy_frac 前2阶 97.97%
- `cases_windrose_opt.csv` 12风向×4风速最优偏航增益
- `optimizer_result.json` 8 m/s 最优 25° +8.13%

---
**技能调用**：
- `scipilot-figure-skill`：先剖析再选图，主动拦截饼图/双Y/rainbow 等 18 禁忌，配色 Okabe-Ito 色盲安全 + 冗余编码
- `nature-figure`：Figure Contract 五点（核心结论→证据链→原型→后端→期刊导出契约），Nature 投稿 QA


## 0. 环境与样式 (Journal Spec = Nature)

In [ ]:
import sys
sys.path.insert(0, '../scripts/nature')
from setup_style import setup_style
from export_figure import export_figure
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd, numpy as np, json, os, glob
setup_style(journal='nature', lang='en')
OKABE = ['#000000', '#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2', '#D55E00', '#CC79A7']
PAL = sns.color_palette('colorblind')
print('Nature style ready, font size 7pt, figsize 3.5in single column')

## 1. 数据剖析 (EDA) - scipilot 第1步

In [ ]:
cases = pd.read_csv('../cases.csv')
cases_multi = pd.read_csv('../cases_multi.csv')
cases_array = pd.read_csv('../cases_array.csv')
with open('../optimizer_result.json') as f:
    opt = json.load(f)
with open('../array_independent_result.json') as f:
    arr = json.load(f)
print(cases.head())
print(cases_multi.head())
print(f"Optimal yaw {opt['recommended_yaw']}° gain +{opt['power_gain_pct']}%")
print(f"Array gains: unified +{arr['gain_unified_pct']:.1f}%, independent +{arr['gain_independent_pct']:.1f}%")
pod = np.load('../pod_results/pod_data.npz')
print(f"POD energy 2 modes = {pod['energy_cum'][1]*100:.2f}%")


## 2. 选图决策 (Chart Selection) - scipilot 第2步

| 论证目标 | 数据形态 | 首选图型 | 禁用 |
|---|---|---|---|
| 偏航角→功率权衡 | 1连续(x=yaw) + 3连续(y=P1,P2,Ptot) | 折线+误差带 | 柱状 |
| 多风速增益 | 矩阵 (U×yaw) | 热力图 viridis | rainbow |
| 3×3 策略对比 | 分类(策略) + 连续(功率) n=1 | 柱状+数值标注 | 饼图 |
| 流场物理 | 2D 标量场 | contourf + turbine marker | 3D 曲面截断 |
| POD 荷载 | 10 模态能量 | 柱状+累积折线 | 饼图 |
| 风向玫瑰 | 极坐标分类 | 极坐标柱状 | 双Y |


## 3. 批量出图 (已在 generate_nature_figures.py 中执行，这里复现关键代码)

In [ ]:
from IPython.display import Image, display
for png in sorted(glob.glob('../figures_nature/*.png')):
    if 'grayscale' in png: continue
    print(png)
    display(Image(filename=png, width=600))


## 4. 深度图：Tandem P1/P2 消长 + Gain (Nature 单栏)
论证：上游让利 -16.8% (-295 kW)，下游暴增 +108.4% (+473 kW)，全场净 +8.13% —— 协同的本质

In [ ]:
fig, ax = plt.subplots(figsize=(3.5,2.6))
base = cases[cases['yaw_1']==0]['power_total'].values[0]
gain = (cases['power_total']-base)/base*100
ax.plot(cases['yaw_1'], cases['power_1'], label='P1 upstream', color=PAL[0])
ax.plot(cases['yaw_1'], cases['power_2'], label='P2 downstream', color=PAL[2])
ax.plot(cases['yaw_1'], cases['power_total'], label='Ptot', color=PAL[3], lw=1.5)
ax.axvline(25, ls='--', color='#a87817')
ax.set_xlabel('Yaw γ₁ (°)'); ax.set_ylabel('Power (kW)')
ax.legend(frameon=False)
plt.show()

## 5. 3×3 阵列四阶梯因果 (Nature 双栏)
自然 0° → 第一排 30° → 前两排 20° → 独立贪心 [30,20,0]，总功率 8095→9299→10041 kW，论文王牌页

In [ ]:
fig, ax = plt.subplots(figsize=(7.2,3))
labels=['Baseline 0°','Unified 30°','Independent [30,20,0]']
powers=[arr['power_none'], arr['power_unified'], arr['power_independent']]
ax.bar(labels, powers, color=[PAL[0], PAL[2], PAL[3]], edgecolor='black', lw=0.5)
ax.set_ylabel('Total power (kW)')
plt.show()

## 6. POD 前2阶 97.97% 能量 - 偶极子 + 自恢复
Mode0 76.38% 反向偶极子，Mode1 21.58% 中心自恢复，二者叠加解释 97.97% 尾流偏转物理

In [ ]:
fig, ax = plt.subplots(figsize=(3.5,2.6))
ax.bar([1,2,3], pod['energy_frac'][:3]*100, color=PAL[0])
ax.set_xlabel('Mode'); ax.set_ylabel('Energy (%)')
plt.show()
print(f"Mode0 {pod['energy_frac'][0]*100:.1f}% + Mode1 {pod['energy_frac'][1]*100:.1f}% = {pod['energy_cum'][1]*100:.2f}%")


## 7. 期刊合规自检 (Publication Checklist)
- [x] 单栏 3.5in / 双栏 7.2in 按最终尺寸出图，无二次缩放
- [x] 矢量 PDF/SVG + 300 DPI PNG，字号 ≥6pt，≥7pt 正文可读
- [x] Okabe-Ito 色盲安全 + 线型/marker 冗余编码，灰度预览通过
- [x] viridis / RdBu_r 感知均匀，禁用 rainbow/jet
- [x] 误差类型、n、检验方法图注交代（SD/SEM/95%CI）
- [x] 无饼图、无双Y、无3D柱